# 11. Optimizer updates — Prodigy, Muon, V4 hybrid Muon, K3 per-head Muon

Small tensors are used only for execution. This notebook no longer labels an ambiguous scaling convention as “generic exact Muon”.
The canonical Muon orthogonalization path and each model-specific scaling rule are named separately.


In [ ]:
import math
import torch

torch.manual_seed(7)
device = torch.device("cpu")


## 1. Prodigy single-parameter reference state chain


In [ ]:
class ProdigyState:
    def __init__(self, parameter, d0=1e-6, beta1=0.9, beta2=0.999):
        self.p0 = parameter.detach().clone()
        self.s = torch.zeros_like(parameter)
        self.exp_avg = torch.zeros_like(parameter)
        self.exp_avg_sq = torch.zeros_like(parameter)
        self.d0 = d0
        self.d = d0
        self.d_max = d0
        self.d_numerator = 0.0
        self.k = 0
        self.beta1 = beta1
        self.beta2 = beta2
        self.beta3 = math.sqrt(beta2)


@torch.no_grad()
def prodigy_step(parameter, gradient, state, lr=1.0, eps=1e-8, weight_decay=0.01):
    d = state.d
    d_lr = d * lr
    state.d_numerator *= state.beta3
    delta = (
        (d / state.d0)
        * d_lr
        * torch.dot(gradient.flatten(), (state.p0 - parameter).flatten()).item()
    )
    state.exp_avg.mul_(state.beta1).add_(gradient, alpha=d * (1 - state.beta1))
    state.exp_avg_sq.mul_(state.beta2).addcmul_(
        gradient,
        gradient,
        value=d * d * (1 - state.beta2),
    )
    state.s.mul_(state.beta3).add_(gradient, alpha=(d / state.d0) * d_lr)
    denominator_for_d = state.s.abs().sum().item()
    if denominator_for_d > 0:
        state.d_numerator += delta
        d_hat = state.d_numerator / denominator_for_d
        state.d_max = max(state.d_max, d_hat)
        state.d = max(state.d, min(state.d_max, float("inf")))
        d = state.d
        d_lr = d * lr

    denom = state.exp_avg_sq.sqrt().add(d * eps)
    parameter.mul_(1 - weight_decay * d_lr)
    parameter.addcdiv_(state.exp_avg, denom, value=-d_lr)
    state.k += 1


parameter = torch.tensor([[1.0, -1.0], [0.5, 2.0]])
state = ProdigyState(parameter)
for scale in (1.0, 0.7, 0.4):
    grad = scale * torch.tensor([[0.2, -0.4], [1.0, 0.5]])
    prodigy_step(parameter, grad, state)
assert state.k == 3


## 2. Muon Newton–Schulz orthogonalization primitive


In [ ]:
def newton_schulz_step(x, coefficients):
    a, b, c = coefficients
    gram = x @ x.mT
    return a * x + (b * gram + c * (gram @ gram)) @ x


def normalize_for_ns(matrix):
    x = matrix.float()
    transposed = x.size(-2) > x.size(-1)
    if transposed:
        x = x.mT
    x = x / (x.norm() + 1e-7)
    return x, transposed


def orthogonalize_muon(matrix, steps=5):
    x, transposed = normalize_for_ns(matrix)
    coefficients = (3.4445, -4.7750, 2.0315)
    for _ in range(steps):
        x = newton_schulz_step(x, coefficients)
    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


def muon_original_scale(rows, columns):
    # Original Muon convention: scale by sqrt(max(1, rows / columns)).
    return math.sqrt(max(1.0, rows / columns))


def muon_original_direction(gradient):
    direction = orthogonalize_muon(gradient)
    rows, columns = gradient.shape
    return muon_original_scale(rows, columns) * direction


g = torch.randn(12, 8)
d = muon_original_direction(g)
assert d.shape == g.shape


## 3. DeepSeek-V4 disclosed hybrid Newton–Schulz path


In [ ]:
def deepseek_v4_hybrid_orthogonalize(matrix):
    x, transposed = normalize_for_ns(matrix)
    aggressive = (3.4445, -4.7750, 2.0315)
    stable = (2.0, -1.5, 0.5)

    for _ in range(8):
        x = newton_schulz_step(x, aggressive)
    for _ in range(2):
        x = newton_schulz_step(x, stable)

    if transposed:
        x = x.mT
    return x.to(matrix.dtype)


@torch.no_grad()
def deepseek_v4_muon_step(
    weight,
    gradient,
    momentum_buffer,
    lr=0.02,
    beta=0.95,
    wd=0.1,
    gamma=0.18,
):
    momentum_buffer.mul_(beta).add_(gradient, alpha=1 - beta)
    nesterov = beta * momentum_buffer + (1 - beta) * gradient
    orthogonal = deepseek_v4_hybrid_orthogonalize(nesterov)
    rows, columns = gradient.shape
    direction = gamma * math.sqrt(max(rows, columns)) * orthogonal
    weight.mul_(1 - lr * wd)
    weight.add_(direction, alpha=-lr)
    return direction


w = torch.randn(16, 12)
g = torch.randn_like(w)
m = torch.zeros_like(w)
assert deepseek_v4_muon_step(w, g, m).shape == w.shape


## 4. Kimi K3 per-head Muon partitioning


In [ ]:
def per_head_muon_direction(gradient, momentum_buffer, num_heads=96, beta=0.95):
    rows, _ = gradient.shape
    assert rows % num_heads == 0
    rows_per_head = rows // num_heads
    updates = []

    for head_index in range(num_heads):
        start = head_index * rows_per_head
        stop = start + rows_per_head
        head_gradient = gradient[start:stop]
        head_momentum = momentum_buffer[start:stop]
        head_momentum.mul_(beta).add_(head_gradient, alpha=1 - beta)
        nesterov = beta * head_momentum + (1 - beta) * head_gradient
        orthogonal = orthogonalize_muon(nesterov)
        scale = math.sqrt(max(head_gradient.shape))
        updates.append(scale * orthogonal)

    return torch.cat(updates, dim=0)


qkv_gradient = torch.randn(96 * 2, 8)
qkv_momentum = torch.zeros_like(qkv_gradient)
update = per_head_muon_direction(qkv_gradient, qkv_momentum)
assert update.shape == qkv_gradient.shape


## Audit result

The previous ambiguous `sqrt(max(rows, cols))` “generic Muon” label is removed. The notebook now distinguishes the original Muon scaling from model-specific V4/K3 conventions.
